# Colab GPU Trong IDE (Git Sync): Train LoRA Minh Khang + Giọng Biểu Cảm + Video Hành Động Wan 2.2

Thiết kế tối ưu cho luồng chạy trực tiếp trong IDE kết nối Google Colab GPU (`L4` khuyên dùng, hoặc `T4`):
1. **Cell 1:** Pull code & ảnh mới nhất từ GitHub (`nguyenkhanhan123/Test`) vào `/content/Test`.
2. **Cell 2:** Train Character LoRA (`nv_minhkhang_lora_v1`) từ 12 ảnh trong `user_inputs/photos/` bằng `ostris/ai-toolkit`.
3. **Cell 3:** Dùng LoRA vừa train sinh Keyframe hành động mới cho `shot_103`, `shot_104`.
4. **Cell 4:** Tạo giọng đọc biểu cảm (Audio-First) trước để khóa độ dài từng cảnh.
5. **Cell 5:** Render cảnh hành động thật bằng `Wan-AI/Wan2.2-TI2V-5B-Diffusers`.
6. **Cell 6:** Ghép video hoàn chỉnh, phát xem ngay trong IDE & tự động ngắt GPU.

In [ ]:
# CELL 1: ĐỒNG BỘ GIT + KIỂM TRA GPU + CHUẨN BỊ CẤU HÌNH LINUX
import os
REPO_URL = 'https://github.com/nguyenkhanhan123/Test.git'
REPO_DIR = '/content/Test'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin main && git reset --hard origin/main

PROJECT_DIR = '/content/Test/ai_film_pipeline'
%cd {PROJECT_DIR}
!nvidia-smi
!pip install -q edge-tts pysubs2 pydantic imageio-ffmpeg scipy pillow numpy mutagen
!python prepare_user_assets.py --trigger nv_minhkhang

In [ ]:
# CELL 2: TRAIN CHARACTER LORA BẰNG OSTRIS/AI-TOOLKIT (SDXL — KHÔNG CẦN HF TOKEN)
!git clone https://github.com/ostris/ai-toolkit.git /content/ai-toolkit || true
!pip install -q -r /content/ai-toolkit/requirements.txt
!python /content/ai-toolkit/run.py {PROJECT_DIR}/user_inputs/train_lora_config.yaml

In [ ]:
# CELL 3: DÙNG LORA VỪA TRAIN SINH KEYFRAME HÀNH ĐỘNG MỚI CHO SHOT_103 & SHOT_104
import glob, json, torch
from pathlib import Path
from diffusers import StableDiffusionXLPipeline

lora_files = sorted(glob.glob(f'{PROJECT_DIR}/loras/nv_minhkhang_lora_v1/*.safetensors'))
assert lora_files, 'Chưa thấy file LoRA .safetensors sau Cell 2!'

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=dtype, use_safetensors=True
).to('cuda')
pipe.load_lora_weights(lora_files[-1])

prompts = {
    'shot_103': 'nv_minhkhang, dynamic martial arts forward dash slash, glowing crimson energy arc, rain sparks, anime film still, wide 16:9',
    'shot_104': 'nv_minhkhang, high-speed mid-air counter strike against glowing red portal energy, dramatic rim lighting, anime film still, wide 16:9',
}
out_dir = Path(PROJECT_DIR) / 'assets' / 'keyframes_lora'
out_dir.mkdir(parents=True, exist_ok=True)
sc_path = Path(PROJECT_DIR) / 'script_action_demo.json'
sc = json.loads(sc_path.read_text(encoding='utf-8'))
for shot in sc['shots']:
    sid = shot['shot_id']
    if sid in prompts:
        img = pipe(prompts[sid], width=1280, height=720, num_inference_steps=28, guidance_scale=7.0).images[0]
        img.save(out_dir / f'{sid}.png')
        shot['keyframe_path'] = f'assets/keyframes_lora/{sid}.png'
        print('Saved LoRA keyframe:', out_dir / f'{sid}.png')
sc_path.write_text(json.dumps(sc, indent=2, ensure_ascii=False), encoding='utf-8')
del pipe
torch.cuda.empty_cache()

In [ ]:
# CELL 4: AUDIO-FIRST — TẠO GIỌNG BIỂU CẢM & KHÓA THỜI LƯỢNG TỪNG SHOT
from pathlib import Path
voices_ready = list((Path(PROJECT_DIR) / 'user_inputs' / 'voices').glob('*.wav'))
if voices_ready:
    !pip install -q git+https://github.com/danhtran2mind/F5-TTS.git whisperx
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='danhtran2mind/Vi-F5-TTS', local_dir='/content/Vi-F5-TTS')
    from cloud_scripts.step3_cloud_vi_f5_tts import clone_episode_voices_vi_f5
    clone_episode_voices_vi_f5(Path(PROJECT_DIR), Path(PROJECT_DIR)/'script_action_demo.json', Path('/content/Vi-F5-TTS'))
else:
    print('Dùng Edge TTS biểu cảm đa sắc thái (thì thầm / căng thẳng / hô chiến đấu / lắng dịu)...')
    !python step_action_demo.py --tts-only

In [ ]:
# CELL 5: RENDER VIDEO HÀNH ĐỘNG THẬT BẰNG WAN 2.2 TI2V-5B
!pip install -q --upgrade git+https://github.com/huggingface/diffusers.git transformers accelerate
!python cloud_scripts/step5_batch_wan22_diffusers.py --script script_action_demo.json --steps 30

In [ ]:
# CELL 6: GHÉP VIDEO HOÀN CHỈNH, PHÁT XEM NGAY TRONG IDE & TỰ ĐỘNG NGẮT GPU
from IPython.display import Video, display

try:
    !python step_action_demo.py --skip-tts
    display(Video(f'{PROJECT_DIR}/output/action_pilot_demo.mp4', embed=True, width=960))
finally:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass